In [3]:
#!/usr/bin/env python3

from pathlib import Path
from collections import Counter
from tabulate import tabulate


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0

    return sum(
        1
        for f in folder.rglob("*")
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS
    )


def analyze_labels(folder: Path):
    class_counter = Counter()
    images_with_labels = 0
    total_instances = 0
    empty_labels = 0
    malformed = 0

    if not folder.exists():
        return class_counter, images_with_labels, total_instances, empty_labels, malformed

    for txt in folder.rglob("*.txt"):
        try:
            lines = [l.strip() for l in txt.read_text().splitlines() if l.strip()]

            if not lines:
                empty_labels += 1
                continue

            images_with_labels += 1

            for line in lines:
                parts = line.split()

                if len(parts) < 5:
                    malformed += 1
                    continue

                try:
                    cls = int(parts[0])
                except ValueError:
                    malformed += 1
                    continue

                class_counter[cls] += 1
                total_instances += 1

        except Exception:
            malformed += 1

    return class_counter, images_with_labels, total_instances, empty_labels, malformed


def main(dataset_root: str):

    dataset_root = Path(dataset_root)

    splits = ["train", "val", "test"]

    global_counter = Counter()

    rows = []

    total_images = 0
    total_instances = 0

    for split in splits:

        images_path = dataset_root / "images" / split
        labels_path = dataset_root / "labels" / split

        num_images = count_images(images_path)

        class_counter, imgs_w_labels, instances, empty, malformed = analyze_labels(labels_path)

        global_counter.update(class_counter)

        total_images += num_images
        total_instances += instances

        rows.append([
            split,
            num_images,
            imgs_w_labels,
            instances,
            empty,
            malformed,
        ])

    print("\nDATASET SUMMARY\n")

    print(tabulate(
        rows,
        headers=[
            "Split",
            "Images",
            "Images w/ Labels",
            "Instances",
            "Empty Labels",
            "Malformed",
        ],
        tablefmt="rounded_grid",
    ))

    print("\nGLOBAL STATS\n")

    print(tabulate([
        ["Total Images", total_images],
        ["Total Instances", total_instances],
        ["Total Classes", len(global_counter)],
    ]))

    print("\nCLASS DISTRIBUTION\n")

    class_rows = []

    for cls, count in sorted(global_counter.items()):
        class_rows.append([cls, count])

    print(tabulate(
        class_rows,
        headers=["Class ID", "Instances"],
        tablefmt="rounded_grid",
    ))


In [4]:

from pathlib import Path
PROJECT_ROOT = Path("")

DATASETS: dict[str, Path] = {
    "FERNANDO": PROJECT_ROOT / "datasets_sdb" / "fernando_ultralytics",
    "KITTI": PROJECT_ROOT / "datasets" / "kitti",
    "AFRICAN_WILDLIFE": PROJECT_ROOT / "datasets" / "african-wildlife",
    "VOC": PROJECT_ROOT / "datasets" / "VOC",
    "HOMEOBJECTS": PROJECT_ROOT / "datasets" / "homeobjects-3K",
     "COCO": PROJECT_ROOT / "datasets" / "coco_subset_5k",
}
main(DATASETS['COCO'])


DATASET SUMMARY

╭─────────┬──────────┬────────────────────┬─────────────┬────────────────┬─────────────╮
│ Split   │   Images │   Images w/ Labels │   Instances │   Empty Labels │   Malformed │
├─────────┼──────────┼────────────────────┼─────────────┼────────────────┼─────────────┤
│ train   │     5000 │               5000 │       21842 │              0 │           0 │
├─────────┼──────────┼────────────────────┼─────────────┼────────────────┼─────────────┤
│ val     │      500 │                500 │        2337 │              0 │           0 │
├─────────┼──────────┼────────────────────┼─────────────┼────────────────┼─────────────┤
│ test    │        0 │                  0 │           0 │              0 │           0 │
╰─────────┴──────────┴────────────────────┴─────────────┴────────────────┴─────────────╯

GLOBAL STATS

---------------  -----
Total Images      5500
Total Instances  24179
Total Classes       12
---------------  -----

CLASS DISTRIBUTION

╭────────────┬─────────────╮
│

In [2]:
!uv add fiftyone

Resolved 293 packages in 1ms
Audited 285 packages in 6ms


In [1]:
import fiftyone as fo
import fiftyone.zoo as foz

# 1. Definir tus 12 categorías exactas mapeadas a los nombres oficiales de COCO
target_classes = [
    "boat", "zebra", "tv", "airplane", "car", "truck", 
    "train", "person", "potted plant", "dog", "cat", "horse"
]

print("Filtrando y descargando el subset de COCO desde el Zoo...")

# 2. Cargar el subset de entrenamiento (train) limitado a 5,000 imágenes
train_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections"],
    classes=target_classes,
    max_samples=5000,      # Limite estricto de 5,000 imágenes
    shuffle=True,          # Mezclar para que sea una muestra aleatoria representativa
    seed=42
)

# 3. Cargar un subset de validación pequeño (ej. 500 imágenes) para que YOLO pueda evaluar
val_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    label_types=["detections"],
    classes=target_classes,
    max_samples=500,
    shuffle=True,
    seed=42
)

# 4. Exportar ambos datasets directamente al formato de Ultralytics YOLO
# Esto creará las carpetas 'images' y 'labels', los .txt normatizados y el 'data.yaml'
print("Exportando a formato Ultralytics YOLO...")

export_dir = "./datasets/coco_subset_5k"

# Exportar entrenamiento
train_dataset.export(
    export_dir=export_dir,
    dataset_type=fo.types.YOLOv5Dataset, # El tipo YOLOv5 es el formato universal .yaml que usa Ultralytics
    split="train",
    classes=target_classes
)

# Exportar validación
val_dataset.export(
    export_dir=export_dir,
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    classes=target_classes
)

print(f"¡Proceso completado! Tu dataset listo para YOLO está en: {export_dir}")

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


Filtrando y descargando el subset de COCO desde el Zoo...
 100% |██████|    1.9Gb/1.9Gb [43.7s elapsed, 0s remaining, 40.2Mb/s]      
Extracting annotations to '/home/josericardopenase/fiftyone/coco-2017/raw/instances_train2017.json'
 100% |████████████████| 5000/5000 [4.8m elapsed, 0s remaining, 15.6 images/s]      
Writing annotations for 5000 downloaded samples to '/home/josericardopenase/fiftyone/coco-2017/train/labels.json'
Dataset info written to '/home/josericardopenase/fiftyone/coco-2017/info.json'
Loading 'coco-2017' split 'train'
 100% |███████████████| 5000/5000 [39.8s elapsed, 0s remaining, 146.0 samples/s]      
Dataset 'coco-2017-train-5000' created
Found annotations at '/home/josericardopenase/fiftyone/coco-2017/raw/instances_val2017.json'
 100% |██████████████████| 500/500 [28.5s elapsed, 0s remaining, 16.8 images/s]      
Writing annotations for 500 downloaded samples to '/home/josericardopenase/fiftyone/coco-2017/validation/labels.json'
Dataset info written to '/home/

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'scissors' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'chair' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'book' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'umbrella' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'dining table' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escr

   2% |\--------------|   79/5000 [311.8ms elapsed, 19.4s remaining, 253.4 samples/s] 

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'backpack' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'donut' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'frisbee' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'kite' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'bicycle' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio

...

   6% |\--------------|  277/5000 [1.6s elapsed, 28.2s remaining, 156.2 samples/s]    

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'microwave' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'orange' not in provided classes
  warnings.warn(msg)
/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'bear' not in provided classes
  warnings.warn(msg)


  39% |█████\---------| 1943/5000 [8.7s elapsed, 13.9s remaining, 213.1 samples/s]    

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/fiftyone/utils/yolo.py:1154: UserWarning: Ignoring object with label 'toaster' not in provided classes
  warnings.warn(msg)


 100% |███████████████| 5000/5000 [21.2s elapsed, 0s remaining, 248.6 samples/s]      
Directory './datasets/coco_subset_5k' already exists; export will be merged with existing files
 100% |█████████████████| 500/500 [2.4s elapsed, 0s remaining, 219.4 samples/s]      
¡Proceso completado! Tu dataset listo para YOLO está en: ./datasets/coco_subset_5k
